# Mine BBH Golden Records (2-Shot vs 0-Shot)
This notebook automatically downloads BigBench Hard (BBH) tasks, formats them into Multiple-Choice (A/B/C/D) style, and mines **Golden Records**—instances where the 2-shot teacher gets the correct answer, but the 0-shot student gets it wrong.

In [1]:
!pip install transformers torch datasets -q

In [2]:
import json
import random
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager" # Disable SDPA for clean hook alignment later
).to(device)
model.eval()
print("Model loaded successfully!")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded successfully!


In [3]:
# Setup MC token IDs and Predict function
A_ID = tokenizer.encode("A", add_special_tokens=False)[0]
B_ID = tokenizer.encode("B", add_special_tokens=False)[0]
C_ID = tokenizer.encode("C", add_special_tokens=False)[0]
D_ID = tokenizer.encode("D", add_special_tokens=False)[0]
MC_IDS = [A_ID, B_ID, C_ID, D_ID]
MC_LETTERS = ["A", "B", "C", "D"]

def format_prompt(text: str) -> str:
    messages = [{"role": "user", "content": text}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) + "Answer:"

def predict_mc(prompt_text: str):
    formatted = format_prompt(prompt_text)
    inputs = tokenizer(formatted, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[:, -1, :].float()
        probs = F.softmax(logits, dim=-1)
    mc_probs = [probs[0, tid].item() for tid in MC_IDS]
    best_idx = mc_probs.index(max(mc_probs))
    return MC_LETTERS[best_idx]

In [4]:
# BBH subsets and their task descriptions
bbh_tasks = {
    'boolean_expressions': "Evaluate the Boolean expression.",
    'navigate': "Determine the final coordinates.",
    'tracking_shuffled_objects_three_objects': "Track the ownership of objects after swaps.",
    'logical_deduction_five_objects': "Deduce the logical ordering.",
    'sports_understanding': "Determine whether the sports statement is plausible or implausible.",
    'date_understanding': "Infer the date from the given context.",
    'snarks': "Identify the sarcastic sentence.",
    'ruin_names': "Identify the humorous edit to the artist or movie name."
}

# Deterministic PRNG for option shuffling
rng = random.Random(42)

def build_mc_options(true_target, unique_targets):
    """
    Takes the true target and dynamically generates 3 wrong targets from the dataset pool.
    Shuffles them into A, B, C, D options.
    """
    pool = [t for t in unique_targets if t != true_target]
    num_wrong = min(3, len(pool))
    wrong_options = rng.sample(pool, num_wrong)
    options = [true_target] + wrong_options
    rng.shuffle(options)
    
    answer_letter = MC_LETTERS[options.index(true_target)]
    options_text = "Options:\n"
    for i, opt in enumerate(options):
        options_text += f"{MC_LETTERS[i]}) {opt}\n"
        
    return options_text, answer_letter

In [5]:
golden_records = []
global_id = 1

for subset, task_desc in bbh_tasks.items():
    print(f"\nDownloading & Processing {subset}...")
    # BBH subsets have exactly 250 examples in the 'test' split
    ds = load_dataset("lukaemon/bbh", subset, split="test")
    all_targets = [ex['target'] for ex in ds]
    unique_targets = list(set(all_targets))
    
    # Use the first 2 examples strictly as the 2-shot demonstrations
    demos = [ds[0], ds[1]]
    
    # Test on the remaining 248 examples
    test_examples = ds.select(range(2, len(ds)))
    
    task_header = f"Task: {task_desc}\n\n"
    demo_str = ""
    for i, ex in enumerate(demos):
        opts_txt, ans_let = build_mc_options(ex['target'], unique_targets)
        demo_str += f"Example {i+1}\n{ex['input']}\n{opts_txt}Answer: {ans_let}\n\n"
        
    for ex in tqdm(test_examples, desc=f"{subset}"):
        opts_txt, ans_let = build_mc_options(ex['target'], unique_targets)
        question_str = f"Question\n{ex['input']}\n{opts_txt.strip()}"
        
        teacher_prompt = task_header + demo_str + question_str
        student_prompt = task_header + question_str
        
        # The dual-pass evaluation loop
        teacher_pred = predict_mc(teacher_prompt)
        student_pred = predict_mc(student_prompt)
        
        teacher_correct = (teacher_pred == ans_let)
        student_correct = (student_pred == ans_let)
        
        # The "Golden" Filter
        if teacher_correct and not student_correct:
            golden_records.append({
                "id": global_id,
                "task": subset,
                "teacher_prompt": teacher_prompt,
                "student_prompt": student_prompt,
                "answer": ans_let
            })
        global_id += 1

print(f"\n--- Search Complete ---")
print(f"Total Golden Records Extracted: {len(golden_records)}")

README.md:   0%|          | 0.00/9.62k [00:00<?, ?B/s]

boolean_expressions/test-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B / 4.52kB            

boolean_expressions/test-00000-of-00001.(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/250 [00:00<?, ? examples/s]

boolean_expressions: 100%|██████████| 248/248 [01:21<00:00,  3.06it/s]


...

ruin_names/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 15.2kB            

ruin_names/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/250 [00:00<?, ? examples/s]

ruin_names: 100%|██████████| 248/248 [03:49<00:00,  1.08it/s]


--- Search Complete ---
Total Golden Records Extracted: 222


In [6]:
# Shuffling the task types to make sure that the nn doesnt get exampless type by type
random.seed(42)
random.shuffle(golden_records)


# Save and preview
with open('bbh_golden_tasks.json', 'w') as f:
    json.dump(golden_records, f, indent=2)
print("Saved golden records to 'bbh_golden_tasks.json'\n")

if golden_records:
    print("--- First Golden Record Preview ---")
    print(f"Answer: {golden_records[0]['answer']}\n")
    print("--- Teacher Prompt ---")
    print(golden_records[0]['teacher_prompt'])
    print("\n--- Student Prompt ---")
    print(golden_records[0]['student_prompt'])

Saved golden records to 'bbh_golden_tasks.json'

--- First Golden Record Preview ---
Answer: C

--- Teacher Prompt ---
Task: Infer the date from the given context.

Example 1
Today is Christmas Eve of 1937. What is the date tomorrow in MM/DD/YYYY?
Options:
(A) 12/11/1937
(B) 12/25/1937
(C) 01/04/1938
(D) 12/04/1937
(E) 12/25/2006
(F) 07/25/1937
Options:
A) (F)
B) (A)
C) (B)
D) (D)
Answer: C

Example 2
In the UK, people usually put the day before the month when formatting the date. Therefore, today is 02/01/1987 to them. What is the date a month ago in MM/DD/YYYY?
Options:
(A) 12/02/1986
(B) 12/01/1986
(C) 03/02/1986
(D) 12/02/2032
(E) 12/02/2062
(F) 02/06/1987
Options:
A) (D)
B) (F)
C) (C)
D) (A)
Answer: D

Question
Jane is celebrating the last day of Jan 2012. What is the date one week ago from today in MM/DD/YYYY?
Options:
(A) 02/06/2012
(B) 08/24/2011
(C) 01/22/2012
(D) 01/24/2012
(E) 01/24/1923
(F) 01/24/1947
Options:
A) (B)
B) (E)
C) (D)
D) (A)

--- Student Prompt ---
Task: Infer 

In [8]:
%run -i mine_bbh_golden_v2.py


Task : boolean_expressions
Mode : dynamic  |  Options : A-B


boolean_expressions: 100%|██████████| 248/248 [01:27<00:00,  2.83it/s]



Task : navigate
Mode : dynamic  |  Options : A-B


navigate: 100%|██████████| 248/248 [03:02<00:00,  1.36it/s]



Task : sports_understanding
Mode : dynamic  |  Options : A-B


...


Task : snarks
Mode : native  |  Options : A-B


snarks: 100%|██████████| 176/176 [01:48<00:00,  1.63it/s]



Task : ruin_names
Mode : native  |  Options : A-D


ruin_names:  39%|███▉      | 97/248 [01:00<01:34,  1.59it/s]


ValueError: Cannot extract native letter from: 'dearth, wind, & fire'

In [10]:
import json
import glob

# 1. Load all records from the 4 files
all_records = []
for fname in glob.glob("bbh_*_v2.json"):
    with open(fname, "r") as f:
        all_records.extend(json.load(f))

# 2. Isolate the "potential" golden records (teacher correct, student wrong)
raw_golden = [
    r for r in all_records 
    if r["teacher_pred"] == r["answer"] and r["student_pred"] != r["answer"]
]

print(f"Total potential Golden Records (no confidence filter): {len(raw_golden)}\n")
print("--- Survival by Teacher Confidence Threshold ---")

# 3. Check survival rates across thresholds
thresholds = [0.60, 0.50, 0.40, 0.30, 0.25, 0.20]

for thresh in thresholds:
    survivors = [r for r in raw_golden if r["teacher_conf"] >= thresh]
    pct = (len(survivors) / len(raw_golden)) * 100 if raw_golden else 0
    print(f"Survive >= {thresh*100:2.0f}% confidence : {len(survivors):>4} records ({pct:.1f}%)")

Total potential Golden Records (no confidence filter): 96

--- Survival by Teacher Confidence Threshold ---
Survive >= 60% confidence :    0 records (0.0%)
Survive >= 50% confidence :    0 records (0.0%)
Survive >= 40% confidence :    0 records (0.0%)
Survive >= 30% confidence :    0 records (0.0%)
Survive >= 25% confidence :    0 records (0.0%)
Survive >= 20% confidence :    0 records (0.0%)


In [11]:
import json, random, glob

# Load all saved records
all_records = []
for fname in glob.glob("bbh_*_v2.json"):
    with open(fname, "r") as f:
        all_records.extend(json.load(f))

# Extract the 96 true golden records
golden = [
    r for r in all_records
    if r["teacher_pred"] == r["answer"] and r["student_pred"] != r["answer"]
]

# Global shuffle
random.seed(42)
random.shuffle(golden)

# Save
with open("bbh_golden_final.json", "w") as f:
    json.dump(golden, f, indent=2)

print(f"Saved {len(golden)} shuffled golden records to bbh_golden_final.json")

from collections import Counter
label_counts = Counter(r["answer"] for r in golden)
print("\nLabel distribution:")
for letter, count in sorted(label_counts.items()):
    pct = 100 * count / len(golden)
    print(f"  {letter} : {count:>3} ({pct:.1f}%)")

task_counts = Counter(r["task"] for r in golden)
print("\nTask distribution:")
for task, count in sorted(task_counts.items()):
    print(f"  {task:<45} : {count:>3}")

Saved 96 shuffled golden records to bbh_golden_final.json

Label distribution:
  A :  61 (63.5%)
  B :   8 (8.3%)
  C :  10 (10.4%)
  D :  13 (13.5%)
  E :   4 (4.2%)

Task distribution:
  boolean_expressions                           :  12
  date_understanding                            :  10
  logical_deduction_five_objects                :  12
  ruin_names                                    :   7
  snarks                                        :   3
  sports_understanding                          :  48
  tracking_shuffled_objects_three_objects       :   4


In [12]:
%run -i augment_golden_permutation.py

Total golden records  : 96
Answer = A            : 61
Answer != A           : 35

Permuting 30 A-records — cycling targets through B, C, D, E ...


Permuting: 100%|██████████| 30/30 [00:14<00:00,  2.04it/s]


Permutation complete:
  Successfully golden after swap : 0
  Teacher failed swap (kept as A): 30

Final golden record count : 96

Label distribution:
  A :  61 (63.5%)
  B :   8 (8.3%)
  C :  10 (10.4%)
  D :  13 (13.5%)
  E :   4 (4.2%)

Task distribution:
  boolean_expressions                           :  12
  date_understanding                            :  10
  logical_deduction_five_objects                :  12
  ruin_names                                    :   7
  snarks                                        :   3
  sports_understanding                          :  48
  tracking_shuffled_objects_three_objects       :   4


In [13]:

import json
import torch
import torch.nn.functional as F
from tqdm import tqdm

# Ensure MC_LETTERS and token IDs are set up
MC_LETTERS_ALL = ["A", "B", "C", "D", "E", "F"]
MC_TOKEN_IDS = {
    letter: tokenizer.encode(letter, add_special_tokens=False)[0]
    for letter in MC_LETTERS_ALL
}

def predict_mc_renormalized(prompt_text: str, valid_letters: list) -> tuple:
    """Returns (predicted_letter, renormalized_confidence)"""
    # Format the prompt exactly as before
    messages = [{"role": "user", "content": prompt_text}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) + "Answer:"
    
    inputs = tokenizer(formatted, return_tensors="pt").to(device)
    
    with torch.no_grad():
        logits = model(**inputs).logits[:, -1, :].float()
        probs = F.softmax(logits, dim=-1) # Softmax over entire vocab
        
    # Extract probabilities for just the valid letters
    valid_probs = [probs[0, MC_TOKEN_IDS[l]].item() for l in valid_letters]
    
    # Renormalize so they sum to 1.0
    sum_valid = sum(valid_probs)
    renorm_probs = [p / sum_valid for p in valid_probs]
    
    # Find the winner
    best_idx = renorm_probs.index(max(renorm_probs))
    winning_letter = valid_letters[best_idx]
    winning_conf = renorm_probs[best_idx]
    
    return winning_letter, winning_conf

# 1. Load the 96 golden records
print("Loading records...")
with open("bbh_golden_final.json", "r") as f:
    golden_records = json.load(f)

# 2. Re-run inference to get proper confidence scores
print(f"Re-evaluating confidence for {len(golden_records)} records...")
for rec in tqdm(golden_records):
    valid_letters = rec["valid_letters"]
    
    # Teacher
    t_pred, t_conf = predict_mc_renormalized(rec["teacher_prompt"], valid_letters)
    # Student
    s_pred, s_conf = predict_mc_renormalized(rec["student_prompt"], valid_letters)
    
    # Update the record
    rec["teacher_pred"] = t_pred
    rec["teacher_conf"] = round(t_conf, 4)
    rec["student_pred"] = s_pred
    rec["student_conf"] = round(s_conf, 4)

# 3. Save the updated records
output_file = "bbh_golden_final_with_conf.json"
with open(output_file, "w") as f:
    json.dump(golden_records, f, indent=2)

print(f"\nSaved updated records to {output_file}")

# 4. Print stats
teacher_confs = [r["teacher_conf"] for r in golden_records]
student_confs = [r["student_conf"] for r in golden_records]

print("\n--- Confidence Summary ---")
print(f"Teacher Average Conf: {sum(teacher_confs)/len(teacher_confs):.2f}")
print(f"Student Average Conf: {sum(student_confs)/len(student_confs):.2f}")
print(f"Teacher Min/Max:      {min(teacher_confs):.2f} / {max(teacher_confs):.2f}")
print(f"Student Min/Max:      {min(student_confs):.2f} / {max(student_confs):.2f}")

print("\n--- Teacher Survival Rates ---")
thresholds = [0.90, 0.80, 0.70, 0.60, 0.50]
for thresh in thresholds:
    survivors = [r for r in golden_records if r["teacher_conf"] >= thresh]
    pct = (len(survivors) / len(golden_records)) * 100
    print(f"Survive >= {thresh*100:2.0f}% confidence : {len(survivors):>3} records ({pct:.1f}%)")

Loading records...
Re-evaluating confidence for 96 records...


100%|██████████| 96/96 [01:07<00:00,  1.42it/s]


Saved updated records to bbh_golden_final_with_conf.json

--- Confidence Summary ---
Teacher Average Conf: 0.50
Student Average Conf: 0.59
Teacher Min/Max:      0.24 / 0.71
Student Min/Max:      0.23 / 0.93

--- Teacher Survival Rates ---
Survive >= 90% confidence :   0 records (0.0%)
Survive >= 80% confidence :   0 records (0.0%)
Survive >= 70% confidence :   1 records (1.0%)
Survive >= 60% confidence :  17 records (17.7%)
Survive >= 50% confidence :  70 records (72.9%)


In [14]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

def load_model():
    return AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        attn_implementation="eager",
    ).to(device)

if 'tokenizer' not in globals():
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
else:
    print("Tokenizer already in memory. Reusing.")

if 'teacher_model' not in globals():
    print("Loading teacher model into memory...")
    teacher_model = load_model()
    for p in teacher_model.parameters():
        p.requires_grad = False
    teacher_model.eval()
else:
    print("Teacher model already in memory. Reusing.")

if 'student_model' not in globals():
    print("Loading student model into memory...")
    student_model = load_model()
    for p in student_model.parameters():
        p.requires_grad = False
    student_model.eval()
else:
    print("Student model already in memory. Reusing.")

Tokenizer already in memory. Reusing.
Loading teacher model into memory...


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Loading student model into memory...


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

In [20]:
import torch
import gc

# Delete all possible model variables if they exist
for var_name in ['model', 'teacher_model', 'student_model']:
    if var_name in globals():
        del globals()[var_name]
        print(f"Deleted {var_name}")

# Force Python garbage collection and empty the GPU cache
gc.collect()
torch.cuda.empty_cache()

print(f"Memory purged! Current allocated VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Deleted teacher_model
Deleted student_model
Memory purged! Current allocated VRAM: 0.16 GB


In [21]:
%run -i train_adapter.py

Loading teacher model into memory...


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Loading student model into memory...


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Injected CalibratedAttention adapter into student model.
Trainable scalars : 393,216
Baseline predictions already cached in memory.
Pre-computing teacher targets for training tasks... (will only run once)


Pre-computing Teacher: 100%|██████████| 80/80 [00:39<00:00,  2.05it/s]


Cached 80 pairs. Layers per task: 25
No checkpoint found. Starting from Epoch 1.

--- Starting Training Chunk: Epoch 1 to 10 ---
Epoch 001 | Loss: 1.5086  CE: 1.3465  Cos: 0.0508
Epoch 002 | Loss: 1.1582  CE: 0.9987  Cos: 0.0458
Epoch 003 | Loss: 0.9876  CE: 0.8301  Cos: 0.0449
Epoch 004 | Loss: 0.8835  CE: 0.7276  Cos: 0.0466
Epoch 005 | Loss: 0.7987  CE: 0.6441  Cos: 0.0506
Epoch 006 | Loss: 0.7289  CE: 0.5753  Cos: 0.0560
Epoch 007 | Loss: 0.6595  CE: 0.5067  Cos: 0.0634
Epoch 008 | Loss: 0.6018  CE: 0.4497  Cos: 0.0721
Epoch 009 | Loss: 0.5537  CE: 0.4022  Cos: 0.0822
Epoch 010 | Loss: 0.5185  CE: 0.3678  Cos: 0.0925

Chunk complete! Saved checkpoint to adapter_epoch_010.pt

--- Evaluation at Epoch 10 ---
ID    Type                           True   Base    Now  Change
-----------------------------------------------------------------


KeyError: 1208

In [24]:
%run -i train_adapter.py

Teacher model already in memory. Reusing.
Student model already in memory. Reusing.
Adapter already injected. Extracted trainable parameters.
Trainable scalars : 393,216
Running baseline (0-shot) on test set...


Baseline Eval:   0%|          | 0/16 [00:00<?, ?it/s]


AttributeError: 'CalibratedAttention' object has no attribute 'q_proj'

In [25]:
golden_records

[{'id': 657,
  'task': 'sports_understanding',
  'teacher_prompt': 'Task: Determine whether the sports statement is plausible or implausible.\n\nExample 1\nIs the following sentence plausible? "Elias Lindholm beat the buzzer."\nOptions:\nA) no\nB) yes\nAnswer: A\n\nExample 2\nIs the following sentence plausible? "John Carlson scored in the third period."\nOptions:\nA) no\nB) yes\nAnswer: B\n\nQuestion\nIs the following sentence plausible? "Gerrit Cole set the hard screen."\nOptions:\nA) no\nB) yes',
  'student_prompt': 'Task: Determine whether the sports statement is plausible or implausible.\n\nQuestion\nIs the following sentence plausible? "Gerrit Cole set the hard screen."\nOptions:\nA) no\nB) yes',
  'answer': 'A',
  'teacher_pred': 'A',
  'teacher_conf': 0.5156,
  'student_pred': 'B',
  'student_conf': 0.5775,
  'valid_letters': ['A', 'B']},
 {'id': 581,
  'task': 'sports_understanding',
  'teacher_prompt': 'Task: Determine whether the sports statement is plausible or implausible.

In [2]:
%run -i train_adapter.py

Device: cuda
Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

...

Loading bbh_golden_final_with_conf.json...
  Loaded 96 records.
Split: 80 train / 16 test
Running 0-shot baseline on test set (pure student, no adapter)...


Baseline: 100%|██████████| 16/16 [00:03<00:00,  4.14it/s]


Baseline accuracy: 0/16
Adapter: freshly injected into student model.
Trainable scalars: 393,216
Pre-computing teacher hidden states (cached after this run)...


Teacher targets: 100%|██████████| 80/80 [00:28<00:00,  2.79it/s]


Cached 80 pairs | 25 hidden-state layers each.
Gradient check: PASSED — U_q/U_k are in the computation graph.
No checkpoint found — starting fresh from Epoch 1.

Training Epochs 1 → 10
Loss = 1.0×CE  +  0.1×LayerCosine(25 layers)  +  0.001×L2

Epoch 001 | Loss: 1.5178  CE: 1.5128  Cos: 0.0500  L2: 157.1
Epoch 002 | Loss: 1.0861  CE: 1.0816  Cos: 0.0448  L2: 157.4
Epoch 003 | Loss: 0.8605  CE: 0.8562  Cos: 0.0432  L2: 158.3
Epoch 004 | Loss: 0.7287  CE: 0.7243  Cos: 0.0441  L2: 159.6
Epoch 005 | Loss: 0.6346  CE: 0.6300  Cos: 0.0467  L2: 161.3
Epoch 006 | Loss: 0.5629  CE: 0.5578  Cos: 0.0505  L2: 163.4
Epoch 007 | Loss: 0.5042  CE: 0.4987  Cos: 0.0553  L2: 165.7
Epoch 008 | Loss: 0.4548  CE: 0.4487  Cos: 0.0610  L2: 168.2
Epoch 009 | Loss: 0.4120  CE: 0.4053  Cos: 0.0677  L2: 170.8
Epoch 010 | Loss: 0.3751  CE: 0.3676  Cos: 0.0753  L2: 173.6

Checkpoint saved → adapter_epoch_010.pt

EVALUATION AT EPOCH 10
ID     Task                               Ans   Base    Now  Change
-------------

In [3]:
%run -i train_adapter.py

Device: cuda
Tokenizer: already in memory.
Teacher model: already in memory.
Student model: already in memory.
Dataset: already in memory (96 records).
Split: 80 train / 16 test
Baseline: already cached in memory.
Adapter: already in model — reusing existing layers.
Trainable scalars: 393,216
Teacher targets: already cached (80 pairs, 25 layers).
Gradient check: PASSED — U_q/U_k are in the computation graph.
Resumed from adapter_epoch_010.pt — starting at Epoch 11.

Training Epochs 11 → 20
Loss = 1.0×CE  +  0.1×LayerCosine(25 layers)  +  0.001×L2

Epoch 011 | Loss: 0.3455  CE: 0.3372  Cos: 0.0832  L2: 176.3
Epoch 012 | Loss: 0.3232  CE: 0.3141  Cos: 0.0910  L2: 179.0
Epoch 013 | Loss: 0.3023  CE: 0.2925  Cos: 0.0980  L2: 181.7
Epoch 014 | Loss: 0.2725  CE: 0.2622  Cos: 0.1034  L2: 184.3
Epoch 015 | Loss: 0.2400  CE: 0.2292  Cos: 0.1082  L2: 187.1
Epoch 016 | Loss: 0.2127  CE: 0.2014  Cos: 0.1133  L2: 189.9
Epoch 017 | Loss: 0.1837  CE: 0.1717  Cos: 0.1197  L2: 192.8
Epoch 018 | Loss: 0

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import json
import random
import re

# ==========================================
# 1. Setup & Un-Inject (NO RELOADING MEMORY)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if 'model' not in globals() or 'tokenizer' not in globals():
    print("Loading base model...")
    model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.bfloat16, attn_implementation="eager"
    ).to(device)
    model.eval()

# Check if model has adapter injected, and revert it to pure base!
was_injected = False
for layer in model.model.layers:
    if hasattr(layer.self_attn, "original_attn"):
        layer.self_attn = layer.self_attn.original_attn
        was_injected = True
if was_injected:
    print("Cleaned adapter out of existing model. Back to Pure Base state!")

class CalibratedAttention(nn.Module):
    def __init__(self, original_attn, rank=4):
        super().__init__()
        self.original_attn = original_attn
        self.config = original_attn.config
        
        target_dtype = original_attn.q_proj.weight.dtype
        
        self.hidden_size = self.config.hidden_size
        self.num_heads = self.config.num_attention_heads
        self.num_kv_heads = self.config.num_key_value_heads
        self.head_dim = self.hidden_size // self.num_heads
        self.num_kv_groups = self.num_heads // self.num_kv_heads
        
        # FIXED: We need to dynamically track what HF expects!
        self._tuple_len = None

        self.U_q = nn.Parameter(torch.randn(self.num_heads, self.head_dim, rank, dtype=target_dtype) * 0.02)
        self.U_k = nn.Parameter(torch.randn(self.num_heads, self.head_dim, rank, dtype=target_dtype) * 0.02)

    def forward(self, hidden_states, **kwargs):
        # Dynamically determine if HF expects 2 or 3 items returned in this exact forward pass
        if self._tuple_len is None:
            with torch.no_grad():
                dummy = self.original_attn(hidden_states=hidden_states, **kwargs)
                self._tuple_len = len(dummy) if isinstance(dummy, tuple) else 1

        bsz, q_len, _ = hidden_states.size()
        
        query_states = self.original_attn.q_proj(hidden_states)
        key_states = self.original_attn.k_proj(hidden_states)
        value_states = self.original_attn.v_proj(hidden_states)

        query_states = query_states.view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        key_states = key_states.view(bsz, q_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        value_states = value_states.view(bsz, q_len, self.num_kv_heads, self.head_dim).transpose(1, 2)

        kv_seq_len = key_states.shape[-2]
        
        # GQA
        key_states = torch.repeat_interleave(key_states, dim=1, repeats=self.num_kv_groups)
        value_states = torch.repeat_interleave(value_states, dim=1, repeats=self.num_kv_groups)

        scale = torch.sqrt(torch.tensor(self.head_dim, dtype=hidden_states.dtype, device=hidden_states.device))
        attn_weights = torch.matmul(query_states, key_states.transpose(2, 3)) / scale

        A = torch.einsum('bhtd,hdr->bhtr', query_states, self.U_q)
        Bm = torch.einsum('bhtd,hdr->bhtr', key_states, self.U_k)
        delta = torch.matmul(A, Bm.transpose(-1, -2))

        attention_mask = kwargs.get("attention_mask", None)
        causal_mask = torch.tril(torch.ones(q_len, kv_seq_len, device=hidden_states.device)).bool()
        attn_weights = attn_weights.masked_fill(~causal_mask, torch.finfo(attn_weights.dtype).min)

        attn_weights = attn_weights + delta

        # Incorporate the external attention_mask (padding mask) if it exists
        if attention_mask is not None:
            attn_weights = attn_weights + attention_mask.to(dtype=attn_weights.dtype)

        attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)
        attn_output = torch.matmul(attn_weights, value_states)
        attn_output = attn_output.transpose(1, 2).contiguous().view(bsz, q_len, self.hidden_size)
        attn_output = self.original_attn.o_proj(attn_output)

        past_key_value = kwargs.get("past_key_value", kwargs.get("past_key_values", None))
        
        # Slice the return tuple exactly so HF doesn't complain
        return (attn_output, past_key_value, None, None)[:self._tuple_len]

# ==========================================
# 2. Prediction Helper
# ==========================================
MC_TOKEN_IDS = {l: tokenizer.encode(l, add_special_tokens=False)[0] for l in ["A","B","C","D","E","F"]}

def predict_mc(prompt_text: str, valid_letters: list) -> tuple:
    messages = [{"role": "user", "content": prompt_text}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) + "Answer:"
    inputs = tokenizer(formatted, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[:, -1, :].float()
        probs  = F.softmax(logits, dim=-1)
    
    mc_probs = [probs[0, MC_TOKEN_IDS[l]].item() for l in valid_letters]
    best_idx = mc_probs.index(max(mc_probs))
    return valid_letters[best_idx], mc_probs[best_idx]

# ==========================================
# 3. Mine 20 Virgin Tasks
# ==========================================
print("Mining 20 completely unseen records from BBH on the fly...")
with open("bbh_golden_final_with_conf.json", "r") as f:
    forbidden_records = json.load(f)
forbidden_prompts = set(r["student_prompt"] for r in forbidden_records)

MC_LETTERS_ALL = ["A", "B", "C", "D", "E", "F"]
rng = random.Random(42)
test_set = []

def extract_native_letter(target: str) -> str:
    m = re.search(r'\(([A-F])\)', target)
    if m: return m.group(1)
    return "A"

def build_dynamic_options(correct_target, wrong_pool, num_options, slot):
    num_wrong = num_options - 1
    sampled_wrong = rng.sample(wrong_pool, min(num_wrong, len(wrong_pool)))
    options = sampled_wrong[:]
    options.insert(slot, correct_target)
    answer_letter  = MC_LETTERS_ALL[slot]
    options_lines  = "\n".join(f"{MC_LETTERS_ALL[i]}) {opt}" for i, opt in enumerate(options))
    return f"Options:\n{options_lines}", answer_letter

TASK_CONFIGS = {
    "sports_understanding": {"desc": "Determine whether the sports statement is plausible or implausible.", "mode": "dynamic", "opts": 2},
    "logical_deduction_five_objects": {"desc": "Deduce the logical ordering.", "mode": "native", "opts": 5}
}

global_id = 1
for subset, cfg in TASK_CONFIGS.items():
    ds = load_dataset("lukaemon/bbh", subset, split="test")
    unique_pool = list(set([ex["target"] for ex in ds])) if cfg["mode"] == "dynamic" else []
    
    demo_str = ""
    slot_counter = 0
    for i in range(2):
        if cfg["mode"] == "dynamic":
            wp = [t for t in unique_pool if t != ds[i]["target"]]
            opts_txt, ans_letter = build_dynamic_options(ds[i]["target"], wp, cfg["opts"], slot_counter % cfg["opts"])
            slot_counter += 1
            demo_str += f"Example {i+1}\n{ds[i]['input']}\n{opts_txt}\nAnswer: {ans_letter}\n\n"
        else:
            ans_letter = extract_native_letter(ds[i]["target"])
            demo_str += f"Example {i+1}\n{ds[i]['input']}\nAnswer: {ans_letter}\n\n"
            
    found_for_task = 0
    for ex in ds.select(range(2, len(ds))):
        if found_for_task >= 10: break
        
        if cfg["mode"] == "dynamic":
            wp = [t for t in unique_pool if t != ex["target"]]
            opts_txt, ans_letter = build_dynamic_options(ex["target"], wp, cfg["opts"], slot_counter % cfg["opts"])
            slot_counter += 1
            question_block = f"Question\n{ex['input']}\n{opts_txt}"
        else:
            ans_letter = extract_native_letter(ex["target"])
            question_block = f"Question\n{ex['input']}"
            
        task_header = f"Task: {cfg['desc']}\n\n"
        student_prompt = task_header + question_block
        teacher_prompt = task_header + demo_str + question_block
        
        if student_prompt not in forbidden_prompts:
            test_set.append({
                "id": global_id,
                "task": subset,
                "student_prompt": student_prompt,
                "teacher_prompt": teacher_prompt,
                "answer": ans_letter,
                "valid_letters": MC_LETTERS_ALL[:cfg["opts"]]
            })
            found_for_task += 1
            global_id += 1

print(f"Successfully mined {len(test_set)} 100% unseen test records.")

# ==========================================
# 4. Run Pure Base & Teacher
# ==========================================
print("\nRunning Phase 1/2: Pure Zero-Shot & Teacher 2-Shot...")
results = []
for t in test_set:
    p0_pred, p0_conf = predict_mc(t["student_prompt"], t["valid_letters"])
    t2_pred, t2_conf = predict_mc(t["teacher_prompt"], t["valid_letters"])
    results.append({
        "id": t["id"], "task": t["task"], "ans": t["answer"],
        "pure_0": (p0_pred, p0_conf), "pure_2": (t2_pred, t2_conf)
    })

# ==========================================
# 5. Inject Adapter & Run Trained 0-Shot
# ==========================================
print("\nInjecting adapter_epoch_020.pt into student...")
trainable_params = []
for layer in model.model.layers:
    cal = CalibratedAttention(layer.self_attn, rank=4).to(device)
    layer.self_attn = cal
    trainable_params.extend([cal.U_q, cal.U_k])

ckpt = torch.load("adapter_epoch_020.pt", map_location=device)
for p, saved_tensor in zip(trainable_params, ckpt["adapter_weights"]):
    p.data.copy_(saved_tensor.to(device=device, dtype=p.dtype))

print("Running Phase 3/3: Calibrated Zero-Shot...")
for i, t in enumerate(test_set):
    cal_pred, cal_conf = predict_mc(t["student_prompt"], t["valid_letters"])
    results[i]["cal_0"] = (cal_pred, cal_conf)

# ==========================================
# 6. Print the Comparative Matrix
# ==========================================
print("\n" + "="*95)
print(f"{'ID':<4} | {'Task':<25} | {'Ans':<3} | {'Pure 0-Shot':<15} | {'Teacher 2-Shot':<15} | {'Trained 0-Shot':<15}")
print("-" * 95)

for r in results:
    task_short = r["task"][:23] + ".." if len(r["task"]) > 25 else r["task"]
    p0_mark = "✅" if r["pure_0"][0] == r["ans"] else "❌"
    t2_mark = "✅" if r["pure_2"][0] == r["ans"] else "❌"
    c0_mark = "✅" if r["cal_0"][0] == r["ans"] else "❌"

    p0_str = f"{r['pure_0'][0]} {p0_mark} ({r['pure_0'][1]:.2f})"
    t2_str = f"{r['pure_2'][0]} {t2_mark} ({r['pure_2'][1]:.2f})"
    c0_str = f"{r['cal_0'][0]} {c0_mark} ({r['cal_0'][1]:.2f})"

    print(f"{r['id']:<4} | {task_short:<25} | {r['ans']:<3} | {p0_str:<15} | {t2_str:<15} | {c0_str:<15}")
print("=" * 95)

Cleaned adapter out of existing model. Back to Pure Base state!
Mining 20 completely unseen records from BBH on the fly...
Successfully mined 20 100% unseen test records.

Running Phase 1/2: Pure Zero-Shot & Teacher 2-Shot...

Injecting adapter_epoch_020.pt into student...
Running Phase 3/3: Calibrated Zero-Shot...

ID   | Task                      | Ans | Pure 0-Shot     | Teacher 2-Shot  | Trained 0-Shot 
-----------------------------------------------------------------------------------------------
1    | sports_understanding      | B   | B ✅ (0.00)      | B ✅ (0.00)      | A ❌ (0.00)     
2    | sports_understanding      | B   | B ✅ (0.00)      | A ❌ (0.00)      | A ❌ (0.00)     
3    | sports_understanding      | B   | B ✅ (0.00)      | B ✅ (0.00)      | A ❌ (0.00)     
4    | sports_understanding      | A   | B ❌ (0.00)      | B ❌ (0.00)      | A ✅ (0.00)     
5    | sports_understanding      | B   | B ✅ (0.00)      | B ✅ (0.00)      | A ❌ (0.00)     
6    | sports_understanding 

In [15]:
import json
from collections import Counter

# Load the dataset
with open("bbh_golden_final_with_conf.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Separate the records logically based on how many options they have
binary_records = [r for r in data if len(r["valid_letters"]) == 2]
multi_choice_records = [r for r in data if len(r["valid_letters"]) > 2]

print(f"Total Records: {len(data)}")
print("-" * 40)

# 1. Overall Distribution
print("OVERALL LABEL DISTRIBUTION:")
overall_counts = Counter(r["answer"] for r in data)
for letter in ["A", "B", "C", "D", "E"]:
    count = overall_counts.get(letter, 0)
    print(f"  {letter}: {count:>2} ({100 * count / len(data):.1f}%)")
print("-" * 40)

# 2. Binary Task Distribution
print(f"BINARY TASKS ONLY (Total: {len(binary_records)}):")
binary_counts = Counter(r["answer"] for r in binary_records)
for letter in ["A", "B", "C", "D", "E"]:
    count = binary_counts.get(letter, 0)
    print(f"  {letter}: {count:>2}")
print("-" * 40)

# 3. Multi-Choice Task Distribution
print(f"MULTI-CHOICE TASKS ONLY (Total: {len(multi_choice_records)}):")
multi_counts = Counter(r["answer"] for r in multi_choice_records)
for letter in ["A", "B", "C", "D", "E"]:
    count = multi_counts.get(letter, 0)
    print(f"  {letter}: {count:>2}")

Total Records: 96
----------------------------------------
OVERALL LABEL DISTRIBUTION:
  A: 31 (32.3%)
  B: 32 (33.3%)
  C: 11 (11.5%)
  D: 11 (11.5%)
  E: 11 (11.5%)
----------------------------------------
BINARY TASKS ONLY (Total: 63):
  A: 31
  B: 32
  C:  0
  D:  0
  E:  0
----------------------------------------
MULTI-CHOICE TASKS ONLY (Total: 33):
  A:  0
  B:  0
  C: 11
  D: 11
  E: 11


Running after data redistribution

In [2]:
%run -i train_adapter.py

Device: cuda
Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

...

Loading bbh_golden_final_with_conf.json...
  Loaded 96 records.
Split: 80 train / 16 test
Running 0-shot baseline on test set (pure student, no adapter)...


Baseline: 100%|██████████| 16/16 [00:03<00:00,  4.09it/s]


Baseline accuracy: 7/16
Adapter: freshly injected into student model.
Trainable scalars: 393,216
Pre-computing teacher hidden states (cached after this run)...


Teacher targets: 100%|██████████| 80/80 [00:32<00:00,  2.47it/s]


Cached 80 pairs | 25 hidden-state layers each.
Gradient check: PASSED — U_q/U_k are in the computation graph.
No checkpoint found — starting fresh from Epoch 1.

Training Epochs 1 → 10
Loss = 1.0×CE  +  0.1×LayerCosine(25 layers)  +  0.001×L2

Epoch 001 | Loss: 1.0689  CE: 1.0638  Cos: 0.0515  L2: 156.4
Epoch 002 | Loss: 0.8726  CE: 0.8676  Cos: 0.0492  L2: 156.6
Epoch 003 | Loss: 0.7727  CE: 0.7677  Cos: 0.0504  L2: 157.3
Epoch 004 | Loss: 0.7016  CE: 0.6963  Cos: 0.0532  L2: 158.4
Epoch 005 | Loss: 0.6281  CE: 0.6224  Cos: 0.0568  L2: 159.9
Epoch 006 | Loss: 0.5583  CE: 0.5522  Cos: 0.0617  L2: 161.7
Epoch 007 | Loss: 0.4960  CE: 0.4892  Cos: 0.0681  L2: 163.7
Epoch 008 | Loss: 0.4447  CE: 0.4371  Cos: 0.0765  L2: 166.0
Epoch 009 | Loss: 0.4047  CE: 0.3959  Cos: 0.0875  L2: 168.5
Epoch 010 | Loss: 0.3812  CE: 0.3711  Cos: 0.1012  L2: 171.3

Checkpoint saved → adapter_epoch_010.pt

EVALUATION AT EPOCH 10
ID     Task                               Ans   Base    Now  Change
-------------